In [87]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [88]:
import os
import qnas_config as cfg
from util import check_files
from cnn.input import GenericDataLoader
from cnn.train_detailed import train_and_eval
from cnn import model, input
import torch
from sklearn.metrics import confusion_matrix
import torch.nn as nn

import medmnist
from medmnist import INFO, Evaluator

In [89]:
phase = 'retrain'
experiment_path = os.path.join("experiments_pathmist", "exp1_repeat_1")
config_file = 'config_files_med/config1.txt'

args = {
    'experiment_path': experiment_path,
    'config_file': config_file,
    'retrain_folder': 'retrai_1',
    'data_path': 'pathmnist_data',
    'dataset': 'pathmnist',
    'log_level': 'INFO',
    'max_epochs': 300,
    'epochs_to_eval': 10,
    'batch_size': 256,
    'eval_batch_size': 256,
    'limit_data': False,
    'num_workers': 4,
    'device': 'cuda:0',
    'lr_scheduler': 'None',
}

In [90]:
check_files(args['experiment_path'])
config = cfg.ConfigParameters(args, phase=phase)
config.get_parameters()

fn_dict=config.fn_dict

In [91]:
config.load_evolved_data(experiment_path=experiment_path)
params = config.train_spec
params

{'available_gpus': [0, 1],
 'batch_size': 256,
 'data_augmentation': True,
 'data_path': 'pathmnist_data',
 'dataset': 'pathmnist',
 'decay': 0.9,
 'device': 'cuda:0',
 'epochs_to_eval': 10,
 'eval_batch_size': 256,
 'experiment_path': 'experiments_pathmist/exp1_repeat_1/retrai_1',
 'learning_rate': 0.001,
 'limit_data': False,
 'limit_data_value': 10000,
 'log_level': 'INFO',
 'max_epochs': 300,
 'mixed_precision': True,
 'momentum': 0.0,
 'num_workers': 4,
 'optimizer': 'AdamW',
 'phase': 'retrain',
 'save_checkpoints_epochs': 10,
 'save_summary_epochs': 0.25,
 'subtract_mean': True,
 'threads': 0,
 'weight_decay': 0.0001,
 'config_file': 'config_files_med/config1.txt',
 'retrain_folder': 'retrai_1',
 'lr_scheduler': 'None'}

In [92]:
evolved_params = config.evolved_params

In [93]:
params['net_list'] = evolved_params['net']
params['fn_dict'] = fn_dict
params['num_classes'] = 9
params['input_shape'] = [128,3, 28, 28]


In [94]:
def reset_and_load_best_model(params, best_model_path, device):
    # Reinitialize the original model
    
    best_model = model.NetworkGraph(num_classes=params["num_classes"], mu=0.99)
    filtered_dict = {key: item for key, item in params['fn_dict'].items() if key in params['net_list']}
    best_model.create_functions(fn_dict=filtered_dict, net_list=params['net_list'])

    input_random = torch.randn(params['input_shape'])
    _ = best_model(input_random)
    # Load the state dictionary of the best model into the new model
    best_model.load_state_dict(torch.load(best_model_path))
    best_model.to(device)

    return best_model

In [95]:
def compute_metrics(model, data_loader, params):
    model.eval()
    all_labels = []
    all_predictions = []
    y_score = torch.tensor([]).to(params['device'])

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(params['device']), labels.to(params['device'])
            y_logits = model(inputs)
            _, predicted = y_logits.max(1)
            output = y_logits.softmax(dim=-1)
            y_score = torch.cat((y_score, output), 0)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
        
        y_score = y_score.cpu().detach().numpy()
    
        evaluator = Evaluator(params['dataset'], split='test', root=params['data_path'])
        metrics = evaluator.evaluate(y_score)
        auc, acc = metrics

    conf_matrix = confusion_matrix(all_labels, all_predictions)
    return conf_matrix, auc, acc

In [96]:
def evaluate(model, criterion, data_loader, params, test=True):
    model.eval()
    eval_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(params['device']), labels.to(params['device'])
            y_logits = model(inputs)
            
            labels = labels.squeeze().long() # medmnist
            loss = criterion(y_logits, labels)
            eval_loss += loss.item()
            _, predicted = y_logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    accuracy = 100 * correct / total
    eval_loss /= len(data_loader)
    
    if test:
        confusion_matrix, auc, acc = compute_metrics(model, data_loader, params)
        return eval_loss, accuracy, auc, acc , confusion_matrix
    return eval_loss, accuracy

In [97]:
data_loader = GenericDataLoader(params=params)

In [98]:
best_model = reset_and_load_best_model(params, os.path.join(experiment_path, 'retrain_2', 'best_model.pth'), args['device'])
criterion = nn.CrossEntropyLoss()

In [99]:
test_loader = data_loader.get_loader(for_train=False, pin_memory_device=args['device'])

In [86]:
evaluated = evaluate(best_model, criterion, test_loader, params, test=True)
evaluated

(0.501867498817115,
 87.25626740947075,
 0.9880057949559458,
 0.8725626740947076,
 array([[1200,    1,    2,    0,    1,  125,    3,    5,    1],
        [   0,  847,    0,    0,    0,    0,    0,    0,    0],
        [   0,    0,  286,    0,    0,   53,    0,    0,    0],
        [   0,    0,    0,  634,    0,    0,    0,    0,    0],
        [  47,    0,   14,   11,  898,   10,   20,   23,   12],
        [   0,    0,   56,    0,    0,  534,    0,    2,    0],
        [   0,    0,    3,   47,    0,    0,  655,    2,   34],
        [   0,    0,   56,   10,    0,  172,   22,  121,   40],
        [   0,    0,   14,   45,    0,    4,   72,    8, 1090]]))